**Question 1**

In [1]:
import pandas as pd, requests
from io import StringIO

In [2]:
url = 'https://en.wikipedia.org/wiki/List_of_S%26P_500_companies'
headers = {'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/91.0.4472.124 Safari/537.36'}
df = pd.read_html(StringIO(requests.get(url, headers=headers).text))[0]

In [3]:
df

,Symbol,Security,GICS Sector,GICS Sub-Industry,Headquarters Location,Date added,CIK,Founded
0,MMM,3M,Industrials,Industrial Conglomerates,"Saint Paul, Minnesota",1957-03-04,66740,1902
1,AOS,A. O. Smith,Industrials,Building Products,"Milwaukee, Wisconsin",2017-07-26,91142,1916
2,ABT,Abbott Laboratories,Health Care,Health Care Equipment,"North Chicago, Illinois",1957-03-04,1800,1888
3,ABBV,AbbVie,Health Care,Biotechnology,"North Chicago, Illinois",2012-12-31,1551152,2013 (1888)
4,ACN,Accenture,Information Technology,IT Consulting & Other Services,"Dublin, Ireland",2011-07-06,1467373,1989
...,...,...,...,...,...,...,...,...
498,XYL,Xylem Inc.,Industrials,Industrial Machinery & Supplies & Components,"White Plains, New York",2011-11-01,1524472,2011
499,YUM,Yum! Brands,Consumer Discretionary,Restaurants,"Louisville, Kentucky",1997-10-06,1041061,1997
500,ZBRA,Zebra Technologies,Information Technology,Electronic Equipment & Instruments,"Lincolnshire, Illinois",2019-12-23,877212,1969
501,ZBH,Zimmer Biomet,Health Care,Health Care Equipment,"Warsaw, Indiana",2001-08-07,1136869,1927


In [4]:
df['Year'] = pd.to_datetime(df['Date added'], errors='coerce').dt.year

df

,Symbol,Security,GICS Sector,GICS Sub-Industry,Headquarters Location,Date added,CIK,Founded,Year
0,MMM,3M,Industrials,Industrial Conglomerates,"Saint Paul, Minnesota",1957-03-04,66740,1902,1957
1,AOS,A. O. Smith,Industrials,Building Products,"Milwaukee, Wisconsin",2017-07-26,91142,1916,2017
2,ABT,Abbott Laboratories,Health Care,Health Care Equipment,"North Chicago, Illinois",1957-03-04,1800,1888,1957
3,ABBV,AbbVie,Health Care,Biotechnology,"North Chicago, Illinois",2012-12-31,1551152,2013 (1888),2012
4,ACN,Accenture,Information Technology,IT Consulting & Other Services,"Dublin, Ireland",2011-07-06,1467373,1989,2011
...,...,...,...,...,...,...,...,...,...
498,XYL,Xylem Inc.,Industrials,Industrial Machinery & Supplies & Components,"White Plains, New York",2011-11-01,1524472,2011,2011
499,YUM,Yum! Brands,Consumer Discretionary,Restaurants,"Louisville, Kentucky",1997-10-06,1041061,1997,1997
500,ZBRA,Zebra Technologies,Information Technology,Electronic Equipment & Instruments,"Lincolnshire, Illinois",2019-12-23,877212,1969,2019
501,ZBH,Zimmer Biomet,Health Care,Health Care Equipment,"Warsaw, Indiana",2001-08-07,1136869,1927,2001


In [5]:
per_year = df[df['Year'] >= 2020]['Year'].value_counts().sort_index()
print(per_year)
print('Highest full year:', per_year[per_year.index <= 2025].idxmax())

Year
2020    10
2021    10
2022    15
2023    15
2024    16
2025    18
2026    13
Name: count, dtype: int64
Highest full year: 2025


In [6]:
# >20 years in index
print('In index >20 years:', (df['Year'] <= 2006).sum())

In index >20 years: 227


**Question 2**

In [7]:
import yfinance as yf
import pandas as pd

In [8]:
indexes = {
    'US - S&P 500': '^GSPC',
    'China - Shanghai': '000001.SS',
    'Hong Kong - Hang Seng': '^HSI',
    'Australia - ASX 200': '^AXJO',
    'India - Nifty 50': '^NSEI',
    'Canada - TSX': '^GSPTSE',
    'Germany - DAX': '^GDAXI',
    'UK - FTSE 100': '^FTSE',
    'Japan - Nikkei 225': '^N225',
    'Mexico - IPC': '^MXX',
    'Brazil - Ibovespa': '^BVSP',
}

In [9]:
results = {}
for name, ticker in indexes.items():
    df = yf.download(ticker, start='2026-01-01', end='2026-08-22',
                     auto_adjust=True, progress=False)
    close = df['Close'][ticker].dropna()
    ytd = (close.iloc[-1] / close.iloc[0] - 1) * 100
    results[name] = float(ytd.iloc[0]) if hasattr(ytd, 'iloc') else float(ytd)

In [10]:
s = pd.Series(results).sort_values(ascending=False)
print(s.round(2))

Japan - Nikkei 225       27.36
Canada - TSX             14.86
US - S&P 500             11.90
UK - FTSE 100             8.70
Brazil - Ibovespa         6.54
Germany - DAX             6.51
Australia - ASX 200       3.79
Mexico - IPC              2.48
Hong Kong - Hang Seng    -1.25
China - Shanghai         -2.94
India - Nifty 50         -7.25
dtype: float64


In [11]:
sp500 = results['US - S&P 500']
better = s[s > sp500]
print(f"\nS&P 500 YTD: {sp500:.2f}%")
print(f"Indexes beating the S&P 500: {len(better)} out of 10")
print(better.round(2))


S&P 500 YTD: 11.90%
Indexes beating the S&P 500: 2 out of 10
Japan - Nikkei 225    27.36
Canada - TSX          14.86
dtype: float64


In [12]:
results_multi = {}
for name, ticker in indexes.items():
    df = yf.download(ticker, start='2015-01-01', end='2026-08-22', auto_adjust=True,
                     progress=False)
    close = df['Close'].dropna()
    end_px = float(close.loc[:'2026-08-21'].iloc[-1, 0])
    for label, anchor in [('3Y', '2023-08-21'), ('5Y', '2021-08-21'), ('10Y', '2016-08-21')]:
        start_px = float(close.loc[:anchor].iloc[-1, 0])
        results_multi.setdefault(name, {})[label] = (end_px / start_px - 1) * 100

In [13]:
table = pd.DataFrame(results_multi).T[['3Y', '5Y', '10Y']].round(2)
print(table)

sp = table.loc['US - S&P 500']
for h in ['3Y', '5Y', '10Y']:
    beat = table.index[(table[h] > sp[h]) & (table.index != 'US - S&P 500')]
    print(f"{h}: {len(beat)} of 10 -> {list(beat)}")

                           3Y      5Y     10Y
US - S&P 500            74.43   72.78  251.41
China - Shanghai        26.26   13.94   25.65
Hong Kong - Hang Seng   47.59    4.67   13.39
Australia - ASX 200     27.31   21.42   63.91
India - Nifty 50        25.05   47.42  179.82
Canada - TSX            85.09   80.05  149.33
Germany - DAX           67.51   65.34  147.87
UK - FTSE 100           49.03   52.61   57.70
Japan - Nikkei 225     109.14  144.39  298.99
Mexico - IPC            23.76   27.84   36.09
Brazil - Ibovespa       49.47   44.88  189.40
3Y: 2 of 10 -> ['Canada - TSX', 'Japan - Nikkei 225']
5Y: 2 of 10 -> ['Canada - TSX', 'Japan - Nikkei 225']
10Y: 1 of 10 -> ['Japan - Nikkei 225']


**Question 3**

In [14]:
import yfinance as yf
import pandas as pd
import numpy as np

In [15]:
# 1. Download S&P 500 daily data from 1950
df = yf.download('^GSPC', start='1950-01-01', end='2026-08-22',
                 auto_adjust=True, progress=False)
close = df['Close']['^GSPC'].dropna()

In [16]:
# 2. Running all-time high, and flag days that set a new ATH
running_max = close.cummax()
is_ath = close >= running_max
ath_dates = close.index[is_ath]

In [17]:
# 3-4. Between each pair of consecutive ATHs, find the trough and its drawdown
corrections = []
for i in range(len(ath_dates) - 1):
    start, nxt = ath_dates[i], ath_dates[i + 1]
    high = close.loc[start]
    window = close.loc[start:nxt]          # from this ATH up to the next
    low = window.min()
    low_date = window.idxmin()
    dd = (high - low) / high * 100
    if dd >= 5:                            # 5. keep only >=5% corrections
        corrections.append({
            'high_date': start,
            'low_date': low_date,
            'high': high,
            'low': low,
            'drawdown_pct': dd,
            'duration_days': (low_date - start).days,   # 6. peak -> trough
        })

corr = pd.DataFrame(corrections)

In [18]:
# Consecutive ATHs each dip slightly below the prior close, so most rows are tiny,
# non-independent wiggles. Collapse runs that share the same peak region:
# a "significant" correction is one bounded by two ATHs with a real trough between.
# The clean way: only count a correction once per peak, which the loop above already
# does, but adjacent ATH pairs still fragment a big decline. See note below.

# 7. Percentiles
for col in ['drawdown_pct', 'duration_days']:
    q = corr[col].quantile([0.25, 0.5, 0.75])
    print(f"\n{col}:")
    print(f"  25th: {q[0.25]:.1f}")
    print(f"  50th (median): {q[0.5]:.1f}")
    print(f"  75th: {q[0.75]:.1f}")

print(f"\nTotal corrections >=5%: {len(corr)}")
print("\nTop 10 by drawdown:")
print(corr.nlargest(10, 'drawdown_pct')[['high_date','low_date','drawdown_pct','duration_days']]
      .to_string(index=False))


drawdown_pct:
  25th: 6.2
  50th (median): 8.0
  75th: 14.0

duration_days:
  25th: 22.0
  50th (median): 40.5
  75th: 86.2

Total corrections >=5%: 74

Top 10 by drawdown:
 high_date   low_date  drawdown_pct  duration_days
2007-10-09 2009-03-09     56.775388            517
2000-03-24 2002-10-09     49.146948            929
1973-01-11 1974-10-03     48.203593            630
1968-11-29 1970-05-26     36.061641            543
2020-02-19 2020-03-23     33.924960             33
1987-08-25 1987-12-04     33.509515            101
1961-12-12 1962-06-26     27.973568            196
1980-11-28 1982-08-12     27.113582            622
2022-01-03 2022-10-12     25.425097            282
1966-02-09 1966-10-07     22.177335            240


**Question 4**

In [19]:
import yfinance as yf
import pandas as pd

In [20]:
ticker = 'AMZN'
ticker_obj = yf.Ticker(ticker)

In [21]:
# 1. Earnings dates (includes Reported EPS, EPS Estimate, Surprise(%))
earnings = ticker_obj.get_earnings_dates()
earnings = earnings.dropna(subset=['Surprise(%)'])   # drop the future row w/ no EPS

In [22]:
# 2. Full daily price history
px = yf.download(ticker, start='2015-01-01', end='2026-08-22',
                 auto_adjust=True, progress=False)
close = px['Close'][ticker].dropna()

In [23]:
# 3. 2-day return over every 3-day window: Close_Day3 / Close_Day1 - 1
#    shift(-2) puts Day3 alongside Day1's row.
ret_2d = close.shift(-2) / close - 1

In [24]:
# --- align earnings dates (timestamps, tz-aware) to trading-day index ---
# get_earnings_dates returns tz-aware datetimes; the price index is tz-naive dates.
earnings = earnings.copy()
earnings['date'] = earnings.index.normalize().tz_localize(None)

In [25]:
# map each earnings announcement to Day1 of its window.
# Day2 is the earnings day, so Day1 = the trading day BEFORE the announcement.
trading_days = close.index
def day1_for(d):
    # last trading day strictly before the earnings date
    prior = trading_days[trading_days < d]
    return prior[-1] if len(prior) else pd.NaT

In [26]:
earnings['day1'] = earnings['date'].apply(day1_for)
earnings['ret_2d'] = earnings['day1'].map(ret_2d) * 100   # in %

In [27]:
# 4. positive surprises -> median 2-day return, and correlation
pos = earnings[earnings['Surprise(%)'] > 0].dropna(subset=['ret_2d'])
print(f"Positive-surprise events: {len(pos)}")
print(f"Median 2-day return after positive surprise: {pos['ret_2d'].median():.2f}%")

Positive-surprise events: 20
Median 2-day return after positive surprise: 0.35%


In [28]:
pos

,EPS Estimate,Reported EPS,Surprise(%),date,day1,ret_2d
Earnings Date,,,,,,
2026-07-30 16:00:00-04:00,1.83,5.75,215.02,2026-07-30,2026-07-29,19.823514
2026-04-29 16:00:00-04:00,1.65,2.78,68.18,2026-04-29,2026-04-28,2.063914
2026-02-05 16:00:00-05:00,1.95,1.95,0.22,2026-02-05,2026-02-04,-9.730030
2025-10-30 16:00:00-04:00,1.56,1.95,25.20,2025-10-30,2025-10-29,6.044289
2025-07-31 16:00:00-04:00,1.33,1.68,26.13,2025-07-31,2025-07-30,-6.707503
2025-05-01 16:00:00-04:00,1.36,1.59,17.08,2025-05-01,2025-04-30,3.014856
2025-02-06 16:00:00-05:00,1.48,1.86,25.36,2025-02-06,2025-02-05,-2.972437
2024-10-31 16:00:00-04:00,1.14,1.43,25.17,2024-10-31,2024-10-30,2.698074
2024-08-01 12:00:00-04:00,1.02,1.26,23.76,2024-08-01,2024-07-31,-10.204301


In [29]:
corr = earnings.dropna(subset=['ret_2d'])[['Surprise(%)', 'ret_2d']].corr()
print("\nCorrelation matrix:")
print(corr.round(3))


Correlation matrix:
             Surprise(%)  ret_2d
Surprise(%)        1.000   0.222
ret_2d             0.222   1.000


In [30]:
spx = yf.download('^GSPC', start='2015-01-01', end='2026-08-22',
                  auto_adjust=True, progress=False)['Close']['^GSPC']
spx_200 = spx.rolling(200).mean()
regime = (spx > spx_200).reindex(close.index).ffill()   # True = bull
earnings['bull'] = earnings['day1'].map(regime)
print(earnings.groupby('bull')['ret_2d'].median())

bull
False    3.014856
True    -0.591301
Name: ret_2d, dtype: float64


**Question 5**

I want to build a volatility forecasting system for the US market, focusing on the S&P 500 and a handful of large-cap stocks over a weekly horizon. I'll target realized volatility rather than returns. I plan to use classical time-series models (ARIMA/SARIMA and GARCH) as the core, benchmarked against a naive random-walk baseline and compared to a machine-learning model, with walk-forward validation and no data leakage. As an application, I'll turn the forecast into a vol-targeting position-sizing overlay evaluated against SPY buy-and-hold, using yfinance price data plus a few FRED macro indicators, with ntfy alerts when forecasted volatility spikes.

**Question 6**

For my volatility-forecasting project, a few useful additions:

**VIX (^VIX, yfinance):** implied volatility, the forward-looking counterpart to my realized-vol target and likely my strongest feature.

**Realized volatility (computed from ^GSPC OHLCV, yfinance):** rolling std of log returns plus a high/low range estimator (Garman-Klass); both my target and key predictor, since vol clusters.

**Term spread (DGS10 minus TB3MS, FRED) and Fed Funds Rate (FEDFUNDS, FRED):** risk-regime indicators, since curves invert and rates shift around the periods when volatility spikes.

Retrieval: `yf.download(['^GSPC','^VIX'], auto_adjust=True)` for prices and `pandas_datareader`'s FRED reader for the macro series. I'd resample the mixed-frequency macro data to weekly and forward-fill only past values to avoid leakage.